In [1]:
pip install --pre -U langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 26.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [2]:
!pip install -U langchain_huggingface

## the model
we will use local model as a judge
we will use "Qwen/Qwen2.5-7B-Instruct"


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LlamaTokenizer ,LlamaForCausalLM ,GenerationConfig ,pipeline ,AutoTokenizer ,AutoModelForCausalLM
import torch
from langchain_huggingface  import HuggingFacePipeline,ChatHuggingFace

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Chatbot Evaluation

In [53]:
from google.colab import userdata
import os
langsmith_api_key = userdata.get('LANGSMITH')

os.environ['LANGCHAIN_TRACING'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = langsmith_api_key


### Define dataset

In [67]:
from langsmith import Client

client = Client()

# define :database : the test case

database_name = "Chatbot evaluation"

dataset = client.create_dataset(
        dataset_name=database_name,
        description="Chatbot evaluation"
                                )

In [68]:
dataset.id

UUID('ea82215c-921b-4fd1-b27e-67791cff3b63')

In [69]:
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs":{"question":"what is langchain"},
            "outputs":{"answer":"A framework for building LLM applications"}
        },
        {
            "inputs":{"question":"What is LangSmith?"},
            "outputs":{"answer":"A platform for observing and evaluating LLM applications"}
        },
        {
            "inputs":{"question":"What is OpenAI?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs":{"question":"What is Google?"},
            "outputs":{"answer":"A technology company known for search"}
        },

        {
            "inputs":{"question":"What is Mistral?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs": {"question": "What is Hugging Face?"},
            "outputs": {"answer": "A company that provides tools and models for natural language processing"}
        },
        {
            "inputs": {"question": "What is TensorFlow?"},
            "outputs": {"answer": "An open-source machine learning framework developed by Google"}
        },
        {
            "inputs": {"question": "What is PyTorch?"},
            "outputs": {"answer": "An open-source machine learning library used for deep learning applications"}
        },
        {
            "inputs": {"question": "What is Anthropic?"},
            "outputs": {"answer": "A company that develops AI systems and large language models"}
        },
        {
            "inputs": {"question": "What is Cohere?"},
            "outputs": {"answer": "A company that provides language AI models and APIs for developers"}
        }

    ]
)

{'example_ids': ['34f477bd-4851-4d03-94b8-aec9bcb87cab',
  'ad945fcc-fef6-4d65-b7df-eded80112fce',
  '6e159eea-e1bd-4671-87e3-561eac9c129a',
  '226e297d-0a6e-4aa5-9ea7-a23b75efd8dc',
  'c6b977eb-93e4-4a56-898f-c6817e316dd5',
  '607fb9d7-e661-4318-8325-0e2d6cbcd86f',
  '78b74080-000a-47a9-ac92-806a81d3fde9',
  'dccf5f40-38c2-4fe4-9ed8-fd8fe1404484',
  '81b7be2c-83e2-4e5c-9ee2-fd8cfca0b412',
  '50f92e52-e693-4f6e-afe4-20debf84c467'],
 'count': 10,
 'as_of': '2026-04-29T14:28:35.518867447Z'}

### Define Metrics (LLM As A Judge)
Qwen/Qwen2.5-7B-Instruct

In [80]:

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading a student's answer.
          Question: {inputs['question']}
          Correct Answer: {reference_outputs['answer']}
          Student Answer: {outputs['response']}

          Instructions:
          - Respond with ONLY one word.
          - Allowed answers: CORRECT or INCORRECT
          - Do not explain.

          grade:"""

      messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]

      text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
      )

      inputs = tokenizer([text], return_tensors="pt").to(model.device)

      outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1
        )

      generated_ids = outputs[0][inputs.input_ids.shape[1]:]

      response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
      )



      return response.strip() == "CORRECT"

In [81]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs:dict, reference_outputs:dict) -> bool:
    response = outputs.get('response', "")
    return int(len(response) < 2 * len(reference_outputs['answer']))

### Run Evaluations

In [82]:
defult_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str ,instructions:str = defult_instructions) -> str:
  messages=[
           {"role": "system", "content": instructions},
           {"role": "user", "content": question}
       ]

  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
  )

  inputs = tokenizer([text], return_tensors="pt").to(model.device)

  outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1
    )

  generated_ids = outputs[0][inputs.input_ids.shape[1]:]

  response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
  )

  return response.strip()

In [83]:
### Call my_app for every datapoints

def ls_target(inputs):
    try:
        result = my_app(inputs['question'])
        return {"response": result}
    except Exception as e:
        return {"response": "", "error": str(e)}


In [84]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target, ## Your AI system
    data = database_name,
    evaluators = [correctness,concision],
    experiment_prefix="Qwen/Qwen2.5-7B-Instruct"

)

View the evaluation results for experiment: 'Qwen/Qwen2.5-7B-Instruct-72a27577' at:
https://smith.langchain.com/o/d43acb99-e975-4cc0-abe8-e96f5723bc0a/datasets/ea82215c-921b-4fd1-b27e-67791cff3b63/compare?selectedSessions=47078b80-7cc0-4c23-8af0-bed578d320cb




0it [00:00, ?it/s]